In [ ]:

"""
Stor hyperparametersökning för responskurvor.

Vad skriptet gör:
- Läser filer av typen CR_q{q}_{curve}_NNLO_GO_450.dat
- Definierar target som medelvärdet av kolumn 2 och 3
- Lägger till respons = 0 från omega = 0 upp till lägsta omega i varje kurva
- Tränar en multi-output-MLP för alla 5 kurvor samtidigt
- Kör 6 leave-one-q-out-folds på de inre q-värdena
- Testar hela parameterrummet som efterfrågats
- Sparar resultat robust i SQLite + CSV så att information finns kvar om körningen kraschar
- Kan återuppta svepet och hoppar över redan klara körningar

Antaganden / tolkningar:
- q-värden: [50, 100, 150, 200, 250, 300, 350, 400]
- 6 folds = validering på de inre q-värdena [100, 150, 200, 250, 300, 350]
  medan q=50 och q=400 alltid ligger i träning.
- dist-features = (q - omega) samt omega / q.
- log-features = log1p(q) och log1p(omega), så att omega=0 fungerar fint.
- weighted MAE använder kurvvis normaliserade vikter så att punkter nära respektive kurvas
  maximum viktas mest, men att små kurvor inte trycks undan av större kurvor.
  Samtidigt ges alla punkter minst vikt 1, så nollrespons-delen tränas också.

Utdata:
- output/sweep_results.sqlite
- output/run_results_latest.csv
- output/curve_metrics_latest.csv
- output/summary_by_config_latest.csv
- output/manifest.json
"""

from __future__ import annotations

import csv
import hashlib
import itertools
import json
import math
import os
import random
import re
import sqlite3
import time
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch import nn


# ============================================================
# 0. Device + seeds
# ============================================================
BASE_SEED = 20260331


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(BASE_SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")


# ============================================================
# 1. Global config
# ============================================================
DATA_ROOT = Path(".")
OUTPUT_DIR = Path("output_fast_fullbatch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUTPUT_DIR / "sweep_results.sqlite"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
RUNS_CSV_PATH = OUTPUT_DIR / "run_results_latest.csv"
CURVE_CSV_PATH = OUTPUT_DIR / "curve_metrics_latest.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "summary_by_config_latest.csv"
LOG_PATH = OUTPUT_DIR / "run_log.txt"

OUTPUT_CURVES = ["R00", "Rt", "Rxy", "Rzz", "R0z"]
CURVE_TO_IDX = {name.lower(): i for i, name in enumerate(OUTPUT_CURVES)}
NUM_OUTPUTS = len(OUTPUT_CURVES)

ALLOWED_QS = [50, 100, 150, 200, 250, 300, 350, 400]
INTERIOR_QS = [100, 150, 200, 250, 300, 350] 
EDGE_QS = [50, 400]

FILE_RE = re.compile(r"^CR_q(\d+)_(R00|Rt|Rxy|Rzz|R0z)_.+\.dat$", re.IGNORECASE)

# Parameterrum
ARCHITECTURES = [
    [64, 64],
    [256, 256],
    [256, 256, 128, 128, 64],
    [128] * 10,
    [128] * 6,
    [128, 128, 64],
]
ACTIVATIONS = ["gelu", "silu", "selu", "tanh"]
OPTIMIZERS = ["adamw"]
LR_POLICIES = ["fixed", "cosine", "plateau_halving"]
BASE_LR = 1e-3
LOSS_NAMES = ["mae", "mse", "weighted_mae"]
NORMALIZATION_OPTIONS = [False, True]
UNIT_SYSTEMS = ["MeV", "GeV"]

FEATURE_SETS = {
    "base": ["q", "omega"],
    "base+dist": ["q", "omega", "q_minus_omega", "omega_over_q"],
    "base+logs": ["q", "omega", "log1p_q", "log1p_omega"],
    "base+dist+logs": [
        "q",
        "omega",
        "q_minus_omega",
        "omega_over_q",
        "log1p_q",
        "log1p_omega",
    ],
}

# Träningsinställningar
MAX_EPOCHS = 700
EARLY_STOP_PATIENCE = 70
MIN_DELTA = 1e-6
BATCH_SIZE = 4096
WEIGHT_DECAY = 1e-4
PLATEAU_PATIENCE = 18
PLATEAU_FACTOR = 0.5
MIN_LR = 1e-6
COSINE_ETA_MIN = 1e-6
WEIGHTED_MAE_ALPHA = 4.0  # viktintervall ~ [1, 5]
WEIGHTED_MAE_POWER = 1.0
EXPORT_EVERY_N_RUNS = 50
MAX_RUNS = None  # sätt till ett heltal för debugging
SAVE_TRAINING_HISTORY = False
HISTORY_DIR = OUTPUT_DIR / "histories"
HISTORY_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Utilities
# ============================================================
def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def atomic_write_text(path: Path, text: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)


def atomic_write_csv(path: Path, fieldnames: List[str], rows: List[dict]) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    os.replace(tmp, path)


def sha1_dict(d: dict) -> str:
    payload = json.dumps(d, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()


def architecture_name(layers: List[int]) -> str:
    return "-".join(str(x) for x in layers)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def safe_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return float("nan")


# ============================================================
# 3. File loading + curve construction
# ============================================================
def is_response_file(path: Path) -> bool:
    return FILE_RE.match(path.name) is not None


def parse_filename(path: Path) -> Tuple[int, str]:
    m = FILE_RE.match(path.name)
    if m is None:
        raise ValueError(f"Ogiltigt filnamn: {path.name}")
    q = int(m.group(1))
    curve = m.group(2)
    return q, curve


def load_single_response_file(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returnerar:
        omega: shape (n,)
        response: shape (n,) = mean(col2, col3) med NaN-tolerans
    """
    arr = np.loadtxt(path)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    # vill ha (n_points, 3)
    if arr.shape[0] == 3 and arr.shape[1] != 3:
        arr = arr.T

    if arr.shape[1] < 3:
        raise ValueError(f"Fil {path.name} måste ha minst 3 kolumner, fick shape={arr.shape}")

    omega = arr[:, 0].astype(np.float64)
    response = np.nanmean(arr[:, 1:3], axis=1).astype(np.float64)
    return omega, response



def fill_leading_nans_with_zero(y: np.ndarray) -> np.ndarray:
    y = y.copy()
    finite = np.isfinite(y)
    if np.any(finite):
        first_finite = int(np.argmax(finite))
        if first_finite > 0:
            y[:first_finite] = 0.0
    else:
        y[:] = 0.0
    return y



def infer_zero_padding_step(omega: np.ndarray) -> float:
    diffs = np.diff(np.sort(np.unique(omega)))
    diffs = diffs[np.isfinite(diffs) & (diffs > 1e-12)]
    if len(diffs) == 0:
        return max(float(np.min(omega)), 1.0)
    return float(np.median(diffs))


@dataclass
class QCurveData:
    q_mev: int
    omega_mev: np.ndarray         # (n_points,)
    y: np.ndarray                 # (n_points, 5)
    weights: np.ndarray           # (n_points, 5)
    peaks: np.ndarray             # (5,)
    inferred_step_mev: float



def compute_relative_curve_weights(y: np.ndarray, alpha: float, power: float) -> Tuple[np.ndarray, np.ndarray]:
    """
    y: (n_points, n_outputs)

    Viktning kurvvis:
        w = 1 + alpha * (|y| / max(|y| per kurva))**power

    Därmed får varje kurva ungefär samma viktintervall oavsett absolut storlek.
    Samtidigt får nollrespons inte vikt 0 utan minst 1.
    """
    peaks = np.max(np.abs(y), axis=0)
    peaks = np.where(peaks < 1e-12, 1.0, peaks)
    rel = np.abs(y) / peaks[None, :]
    weights = 1.0 + alpha * np.power(rel, power)
    return weights.astype(np.float64), peaks.astype(np.float64)



def build_q_curve_data(data_root: Path) -> Dict[int, QCurveData]:
    files = sorted([p for p in data_root.glob("*.dat") if is_response_file(p)])
    if not files:
        raise FileNotFoundError(
            f"Hittade inga responsfiler i {data_root.resolve()}. "
            f"Förväntade namn som CR_q50_R00_NNLO_GO_450.dat"
        )

    grouped: Dict[int, Dict[str, Tuple[np.ndarray, np.ndarray]]] = {}
    for path in files:
        q, curve = parse_filename(path)
        if q not in ALLOWED_QS:
            continue
        omega, response = load_single_response_file(path)
        grouped.setdefault(q, {})[curve] = (omega, response)

    missing_qs = [q for q in ALLOWED_QS if q not in grouped]
    if missing_qs:
        raise ValueError(f"Följande q-värden saknas i datan: {missing_qs}")

    q_data: Dict[int, QCurveData] = {}

    for q in ALLOWED_QS:
        curves = grouped[q]
        missing_curves = [c for c in OUTPUT_CURVES if c not in curves]
        if missing_curves:
            raise ValueError(f"q={q} saknar kurvor: {missing_curves}")

        omega_ref = None
        y_cols = []

        for curve_name in OUTPUT_CURVES:
            omega, y = curves[curve_name]
            y = fill_leading_nans_with_zero(y)

            if omega_ref is None:
                omega_ref = omega.copy()
            else:
                if len(omega) != len(omega_ref) or not np.allclose(omega, omega_ref, rtol=0.0, atol=1e-9):
                    raise ValueError(
                        f"Omega-grid skiljer sig mellan kurvor för q={q}. "
                        "Skriptet antar samma omega-grid för alla 5 kurvor."
                    )

            y_cols.append(y)

        omega_ref = np.asarray(omega_ref, dtype=np.float64)
        y_mat = np.stack(y_cols, axis=1)  # (n_points, 5)

        mask = np.isfinite(omega_ref) & np.all(np.isfinite(y_mat), axis=1)
        omega_clean = omega_ref[mask]
        y_clean = y_mat[mask]

        if len(omega_clean) == 0:
            raise ValueError(f"Inga giltiga datapunkter kvar för q={q}")

        omega_clean = np.asarray(omega_clean, dtype=np.float64)
        y_clean = np.asarray(y_clean, dtype=np.float64)

        step = infer_zero_padding_step(omega_clean)
        omega_min = float(np.min(omega_clean))

        if omega_min > 1e-12:
            omega_zeros = np.arange(0.0, omega_min, step, dtype=np.float64)
            # säkerställ att vi inte duplicerar första riktiga omega
            omega_zeros = omega_zeros[omega_zeros < omega_min - 1e-12]
        else:
            omega_zeros = np.empty((0,), dtype=np.float64)

        y_zeros = np.zeros((len(omega_zeros), NUM_OUTPUTS), dtype=np.float64)

        omega_aug = np.concatenate([omega_zeros, omega_clean], axis=0)
        y_aug = np.concatenate([y_zeros, y_clean], axis=0)

        weights, peaks = compute_relative_curve_weights(
            y_aug,
            alpha=WEIGHTED_MAE_ALPHA,
            power=WEIGHTED_MAE_POWER,
        )

        q_data[q] = QCurveData(
            q_mev=q,
            omega_mev=omega_aug,
            y=y_aug,
            weights=weights,
            peaks=peaks,
            inferred_step_mev=step,
        )

    return q_data


# ============================================================
# 4. Folds
# ============================================================
def build_6_folds() -> List[dict]:
    folds = []
    for i, val_q in enumerate(INTERIOR_QS, start=1):
        train_qs = sorted([q for q in ALLOWED_QS if q != val_q])
        folds.append(
            {
                "fold_name": f"fold_{i:02d}",
                "fold_index": i,
                "val_q": val_q,
                "train_qs": train_qs,
            }
        )
    return folds


# ============================================================
# 5. Features + data manager
# ============================================================
def convert_energy(x_mev: float, unit_system: str) -> float:
    if unit_system == "MeV":
        return float(x_mev)
    if unit_system == "GeV":
        return float(x_mev) / 1000.0
    raise ValueError(f"Okänt enhetssystem: {unit_system}")



def build_feature_vector(q_mev: float, omega_mev: float, feature_names: List[str], unit_system: str) -> List[float]:
    q = convert_energy(q_mev, unit_system)
    omega = convert_energy(omega_mev, unit_system)
    eps = 1e-12

    values = {
        "q": q,
        "omega": omega,
        "q_minus_omega": q - omega,
        "omega_over_q": 0.0 if abs(q) < eps else omega / q,
        "log1p_q": math.log1p(max(q, 0.0)),
        "log1p_omega": math.log1p(max(omega, 0.0)),
    }
    return [float(values[name]) for name in feature_names]


class FoldDataManager:
    def __init__(
        self,
        q_data: Dict[int, QCurveData],
        feature_set_name: str,
        normalize: bool,
        unit_system: str,
        device: torch.device,
    ):
        self.q_data = q_data
        self.feature_set_name = feature_set_name
        self.feature_names = FEATURE_SETS[feature_set_name]
        self.normalize = bool(normalize)
        self.unit_system = unit_system
        self.device = device

        self.x_mean = None
        self.x_std = None
        self.y_mean = None
        self.y_std = None

    def _collect_for_qs(self, q_list: List[int]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        xs, ys, ws, q_ids = [], [], [], []
        for q in q_list:
            pack = self.q_data[q]
            for i in range(len(pack.omega_mev)):
                x = build_feature_vector(q, float(pack.omega_mev[i]), self.feature_names, self.unit_system)
                xs.append(x)
                ys.append(pack.y[i].tolist())
                ws.append(pack.weights[i].tolist())
                q_ids.append(q)

        X = np.asarray(xs, dtype=np.float32)
        Y = np.asarray(ys, dtype=np.float32)
        W = np.asarray(ws, dtype=np.float32)
        QID = np.asarray(q_ids, dtype=np.int32)
        return X, Y, W, QID

    def configure(self, train_qs: List[int], val_qs: List[int]) -> None:
        X_train_raw, Y_train_raw, W_train, Q_train = self._collect_for_qs(train_qs)
        X_val_raw, Y_val_raw, W_val, Q_val = self._collect_for_qs(val_qs)

        X_train_raw = torch.tensor(X_train_raw, dtype=torch.float32, device=self.device)
        Y_train_raw = torch.tensor(Y_train_raw, dtype=torch.float32, device=self.device)
        W_train = torch.tensor(W_train, dtype=torch.float32, device=self.device)

        X_val_raw = torch.tensor(X_val_raw, dtype=torch.float32, device=self.device)
        Y_val_raw = torch.tensor(Y_val_raw, dtype=torch.float32, device=self.device)
        W_val = torch.tensor(W_val, dtype=torch.float32, device=self.device)

        if self.normalize:
            self.x_mean = X_train_raw.mean(dim=0, keepdim=True)
            self.x_std = X_train_raw.std(dim=0, keepdim=True)
            self.y_mean = Y_train_raw.mean(dim=0, keepdim=True)
            self.y_std = Y_train_raw.std(dim=0, keepdim=True)

            self.x_std = torch.where(self.x_std < 1e-12, torch.ones_like(self.x_std), self.x_std)
            self.y_std = torch.where(self.y_std < 1e-12, torch.ones_like(self.y_std), self.y_std)
        else:
            self.x_mean = torch.zeros((1, X_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.x_std = torch.ones((1, X_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.y_mean = torch.zeros((1, Y_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.y_std = torch.ones((1, Y_train_raw.shape[1]), dtype=torch.float32, device=self.device)

        self.X_train = self.x_to_model_space(X_train_raw)
        self.Y_train_raw = Y_train_raw
        self.W_train = W_train
        self.Q_train = Q_train

        self.X_val = self.x_to_model_space(X_val_raw)
        self.Y_val_raw = Y_val_raw
        self.W_val = W_val
        self.Q_val = Q_val

    def x_to_model_space(self, X_raw: torch.Tensor) -> torch.Tensor:
        return (X_raw - self.x_mean) / self.x_std

    def y_from_model_space(self, Y_model: torch.Tensor) -> torch.Tensor:
        return Y_model * self.y_std + self.y_mean

    def dataset_for_single_q(self, q: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        pack = self.q_data[q]
        X = np.asarray(
            [build_feature_vector(q, float(w), self.feature_names, self.unit_system) for w in pack.omega_mev],
            dtype=np.float32,
        )
        Y = pack.y.astype(np.float32)
        W = pack.weights.astype(np.float32)

        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
        W_t = torch.tensor(W, dtype=torch.float32, device=self.device)
        X_t = self.x_to_model_space(X_t)
        return X_t, Y_t, W_t



def build_cached_fold_managers(
    q_data: Dict[int, QCurveData],
    folds: List[dict],
    device: torch.device,
) -> Dict[Tuple[str, bool, str, int], FoldDataManager]:
    """
    Bygger alla FoldDataManager exakt en gång per
    (feature_set, normalize, unit_system, fold_index).

    Det ändrar inte träningsmålet eller sweepen, men eliminerar en stor mängd
    upprepad preprocessing mellan körningarna.
    """
    cache: Dict[Tuple[str, bool, str, int], FoldDataManager] = {}
    for feature_set in FEATURE_SETS.keys():
        for normalize in NORMALIZATION_OPTIONS:
            for unit_system in UNIT_SYSTEMS:
                for fold in folds:
                    dm = FoldDataManager(
                        q_data=q_data,
                        feature_set_name=feature_set,
                        normalize=normalize,
                        unit_system=unit_system,
                        device=device,
                    )
                    dm.configure(fold["train_qs"], [fold["val_q"]])
                    cache[(feature_set, bool(normalize), unit_system, fold["fold_index"])] = dm
    return cache


# ============================================================
# 6. Model + loss + metrics
# ============================================================
def make_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "gelu":
        return nn.GELU()
    if name == "silu":
        return nn.SiLU()
    if name == "selu":
        return nn.SELU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Okänd activation: {name}")


class MultiOutputMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_layers: List[int], output_dim: int, activation: str):
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for hidden in hidden_layers:
            layers.append(nn.Linear(prev, hidden))
            layers.append(make_activation(activation))
            prev = hidden
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)



def mae_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(y_pred_raw - y_true_raw))



def mse_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor) -> torch.Tensor:
    return torch.mean((y_pred_raw - y_true_raw) ** 2)



def weighted_mae_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    err = torch.abs(y_pred_raw - y_true_raw)
    per_curve = (weights * err).sum(dim=0) / (weights.sum(dim=0) + 1e-12)
    return per_curve.mean()



def objective_value(loss_name: str, y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    if loss_name == "mae":
        return mae_loss_raw(y_pred_raw, y_true_raw)
    if loss_name == "mse":
        return mse_loss_raw(y_pred_raw, y_true_raw)
    if loss_name == "weighted_mae":
        return weighted_mae_loss_raw(y_pred_raw, y_true_raw, weights)
    raise ValueError(f"Okänd loss_name: {loss_name}")



def evaluate_tensor(y_true: torch.Tensor, y_pred: torch.Tensor, weights: torch.Tensor) -> dict:
    err = y_pred - y_true
    abs_err = torch.abs(err)
    sq_err = err ** 2

    mae = torch.mean(abs_err).item()
    mse = torch.mean(sq_err).item()
    per_curve_wmae = (weights * abs_err).sum(dim=0) / (weights.sum(dim=0) + 1e-12)
    wmae = per_curve_wmae.mean().item()

    per_curve_mae = torch.mean(abs_err, dim=0).detach().cpu().numpy()
    per_curve_mse = torch.mean(sq_err, dim=0).detach().cpu().numpy()
    per_curve_wmae_np = per_curve_wmae.detach().cpu().numpy()
    peaks = torch.max(torch.abs(y_true), dim=0).values.detach().cpu().numpy()

    return {
        "mae": float(mae),
        "mse": float(mse),
        "weighted_mae": float(wmae),
        "per_curve_mae": per_curve_mae,
        "per_curve_mse": per_curve_mse,
        "per_curve_weighted_mae": per_curve_wmae_np,
        "per_curve_peak_abs": peaks,
        "n_points": int(y_true.shape[0]),
    }



def evaluate_model_on_dataset(model: nn.Module, dm: FoldDataManager, X: torch.Tensor, Y_raw: torch.Tensor, W: torch.Tensor) -> dict:
    model.eval()
    with torch.no_grad():
        pred_model = model(X)
        pred_raw = dm.y_from_model_space(pred_model)
        return evaluate_tensor(Y_raw, pred_raw, W)


# ============================================================
# 7. Scheduler helpers
# ============================================================
def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)



def build_scheduler(optimizer: torch.optim.Optimizer, lr_policy: str):
    if lr_policy == "fixed":
        return None
    if lr_policy == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=MAX_EPOCHS,
            eta_min=COSINE_ETA_MIN,
        )
    if lr_policy == "plateau_halving":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=PLATEAU_FACTOR,
            patience=PLATEAU_PATIENCE,
            threshold=MIN_DELTA,
            threshold_mode="abs",
            min_lr=MIN_LR,
        )
    raise ValueError(f"Okänd lr_policy: {lr_policy}")


# ============================================================
# 8. Training
# ============================================================
@dataclass
class RunConfig:
    fold_name: str
    fold_index: int
    val_q: int
    train_qs: List[int]
    architecture: List[int]
    activation: str
    optimizer: str
    lr_policy: str
    base_lr: float
    loss_name: str
    feature_set: str
    normalize: bool
    unit_system: str

    def run_id(self) -> str:
        return sha1_dict(asdict(self))[:16]



def train_one_run(dm: FoldDataManager, cfg: RunConfig) -> dict:
    input_dim = len(FEATURE_SETS[cfg.feature_set])
    model = MultiOutputMLP(
        input_dim=input_dim,
        hidden_layers=cfg.architecture,
        output_dim=NUM_OUTPUTS,
        activation=cfg.activation,
    ).to(DEVICE)

    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer, cfg.lr_policy)

    n_train = dm.X_train.shape[0]
    best_state = None
    best_metrics = None
    best_epoch = -1
    best_objective = float("inf")
    epochs_without_improvement = 0
    history = []

    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        perm = torch.randperm(n_train, device=DEVICE)
        xb = dm.X_train[perm]
        yb = dm.Y_train_raw[perm]
        wb = dm.W_train[perm]

        optimizer.zero_grad(set_to_none=True)
        pred_model = model(xb)
        pred_raw = dm.y_from_model_space(pred_model)
        loss = objective_value(cfg.loss_name, pred_raw, yb, wb)
        loss.backward()
        optimizer.step()

        val_metrics = evaluate_model_on_dataset(model, dm, dm.X_val, dm.Y_val_raw, dm.W_val)
        current_objective = float(val_metrics[cfg.loss_name])
        current_lr = float(optimizer.param_groups[0]["lr"])
        epoch_train_loss = float(loss.item())

        history.append(
            {
                "epoch": epoch,
                "train_objective": epoch_train_loss,
                "val_mae": val_metrics["mae"],
                "val_mse": val_metrics["mse"],
                "val_weighted_mae": val_metrics["weighted_mae"],
                "lr": current_lr,
            }
        )

        improved = (best_objective - current_objective) > MIN_DELTA
        if np.isfinite(current_objective) and improved:
            best_objective = current_objective
            best_epoch = epoch
            best_metrics = val_metrics
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if cfg.lr_policy == "plateau_halving" and scheduler is not None:
            scheduler.step(current_objective)
        elif cfg.lr_policy == "cosine" and scheduler is not None:
            scheduler.step()

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_metrics = evaluate_model_on_dataset(model, dm, dm.X_val, dm.Y_val_raw, dm.W_val)
    runtime_sec = time.time() - t0
    final_lr = float(optimizer.param_groups[0]["lr"])

    result = {
        "model": model,
        "best_epoch": int(best_epoch),
        "epochs_ran": int(len(history)),
        "best_objective": float(best_objective),
        "best_metrics": best_metrics if best_metrics is not None else final_val_metrics,
        "final_val_metrics": final_val_metrics,
        "final_lr": final_lr,
        "runtime_sec": float(runtime_sec),
        "history": history,
        "num_params": int(count_parameters(model)),
    }
    return result


# ============================================================
# 9. SQLite persistence
# ============================================================
def get_db_connection(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    return conn



def init_db(conn: sqlite3.Connection) -> None:
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS runs (
            run_id TEXT PRIMARY KEY,
            status TEXT NOT NULL,
            error_message TEXT,
            traceback_text TEXT,
            fold_name TEXT,
            fold_index INTEGER,
            val_q INTEGER,
            train_qs TEXT,
            architecture TEXT,
            activation TEXT,
            optimizer TEXT,
            lr_policy TEXT,
            base_lr REAL,
            loss_name TEXT,
            feature_set TEXT,
            normalize INTEGER,
            unit_system TEXT,
            seed INTEGER,
            train_size INTEGER,
            val_size INTEGER,
            num_params INTEGER,
            best_epoch INTEGER,
            epochs_ran INTEGER,
            best_objective REAL,
            val_mae REAL,
            val_mse REAL,
            val_weighted_mae REAL,
            final_lr REAL,
            runtime_sec REAL,
            started_at REAL,
            finished_at REAL
        )
        """
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS curve_metrics (
            run_id TEXT,
            curve_name TEXT,
            val_mae REAL,
            val_mse REAL,
            val_weighted_mae REAL,
            peak_abs REAL,
            PRIMARY KEY (run_id, curve_name)
        )
        """
    )
    conn.commit()



def run_completed(conn: sqlite3.Connection, run_id: str) -> bool:
    row = conn.execute(
        "SELECT status FROM runs WHERE run_id = ?",
        (run_id,),
    ).fetchone()
    return row is not None and row[0] == "completed"



def upsert_run_start(conn: sqlite3.Connection, cfg: RunConfig, seed: int, train_size: int, val_size: int) -> None:
    conn.execute(
        """
        INSERT INTO runs (
            run_id, status, error_message, traceback_text,
            fold_name, fold_index, val_q, train_qs,
            architecture, activation, optimizer, lr_policy, base_lr,
            loss_name, feature_set, normalize, unit_system, seed,
            train_size, val_size, started_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(run_id) DO UPDATE SET
            status=excluded.status,
            error_message=excluded.error_message,
            traceback_text=excluded.traceback_text,
            started_at=excluded.started_at,
            finished_at=NULL
        """,
        (
            cfg.run_id(),
            "running",
            None,
            None,
            cfg.fold_name,
            cfg.fold_index,
            cfg.val_q,
            ",".join(str(x) for x in cfg.train_qs),
            architecture_name(cfg.architecture),
            cfg.activation,
            cfg.optimizer,
            cfg.lr_policy,
            cfg.base_lr,
            cfg.loss_name,
            cfg.feature_set,
            int(cfg.normalize),
            cfg.unit_system,
            seed,
            train_size,
            val_size,
            time.time(),
        ),
    )
    conn.commit()



def upsert_run_failure(conn: sqlite3.Connection, cfg: RunConfig, error_message: str, tb: str) -> None:
    conn.execute(
        """
        UPDATE runs
        SET status = ?, error_message = ?, traceback_text = ?, finished_at = ?
        WHERE run_id = ?
        """,
        (
            "failed",
            error_message[:4000],
            tb[:50000],
            time.time(),
            cfg.run_id(),
        ),
    )
    conn.commit()



def upsert_run_success(conn: sqlite3.Connection, cfg: RunConfig, result: dict) -> None:
    metrics = result["final_val_metrics"]
    conn.execute(
        """
        UPDATE runs
        SET status = ?,
            error_message = NULL,
            traceback_text = NULL,
            num_params = ?,
            best_epoch = ?,
            epochs_ran = ?,
            best_objective = ?,
            val_mae = ?,
            val_mse = ?,
            val_weighted_mae = ?,
            final_lr = ?,
            runtime_sec = ?,
            finished_at = ?
        WHERE run_id = ?
        """,
        (
            "completed",
            result["num_params"],
            result["best_epoch"],
            result["epochs_ran"],
            result["best_objective"],
            metrics["mae"],
            metrics["mse"],
            metrics["weighted_mae"],
            result["final_lr"],
            result["runtime_sec"],
            time.time(),
            cfg.run_id(),
        ),
    )

    for curve_idx, curve_name in enumerate(OUTPUT_CURVES):
        conn.execute(
            """
            INSERT INTO curve_metrics (run_id, curve_name, val_mae, val_mse, val_weighted_mae, peak_abs)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(run_id, curve_name) DO UPDATE SET
                val_mae=excluded.val_mae,
                val_mse=excluded.val_mse,
                val_weighted_mae=excluded.val_weighted_mae,
                peak_abs=excluded.peak_abs
            """,
            (
                cfg.run_id(),
                curve_name,
                float(metrics["per_curve_mae"][curve_idx]),
                float(metrics["per_curve_mse"][curve_idx]),
                float(metrics["per_curve_weighted_mae"][curve_idx]),
                float(metrics["per_curve_peak_abs"][curve_idx]),
            ),
        )
    conn.commit()


# ============================================================
# 10. Export from DB
# ============================================================
def export_runs_csv(conn: sqlite3.Connection, path: Path) -> None:
    rows = conn.execute(
        "SELECT * FROM runs ORDER BY status DESC, finished_at ASC, run_id ASC"
    ).fetchall()
    cols = [x[0] for x in conn.execute("SELECT * FROM runs LIMIT 1").description]
    rows_dict = [dict(zip(cols, row)) for row in rows]
    if rows_dict:
        atomic_write_csv(path, cols, rows_dict)



def export_curve_csv(conn: sqlite3.Connection, path: Path) -> None:
    rows = conn.execute(
        "SELECT * FROM curve_metrics ORDER BY run_id ASC, curve_name ASC"
    ).fetchall()
    cols = [x[0] for x in conn.execute("SELECT * FROM curve_metrics LIMIT 1").description]
    rows_dict = [dict(zip(cols, row)) for row in rows]
    if rows_dict:
        atomic_write_csv(path, cols, rows_dict)



def export_summary_csv(conn: sqlite3.Connection, path: Path) -> None:
    rows = conn.execute(
        """
        SELECT
            architecture,
            activation,
            optimizer,
            lr_policy,
            base_lr,
            loss_name,
            feature_set,
            normalize,
            unit_system,
            COUNT(*) AS n_completed_folds,
            AVG(val_mae) AS mean_val_mae,
            AVG(val_mse) AS mean_val_mse,
            AVG(val_weighted_mae) AS mean_val_weighted_mae,
            AVG(best_epoch) AS mean_best_epoch,
            AVG(final_lr) AS mean_final_lr,
            AVG(runtime_sec) AS mean_runtime_sec
        FROM runs
        WHERE status = 'completed'
        GROUP BY
            architecture, activation, optimizer, lr_policy, base_lr,
            loss_name, feature_set, normalize, unit_system
        ORDER BY mean_val_mae ASC, mean_val_mse ASC, mean_val_weighted_mae ASC
        """
    ).fetchall()

    cols = [
        "architecture",
        "activation",
        "optimizer",
        "lr_policy",
        "base_lr",
        "loss_name",
        "feature_set",
        "normalize",
        "unit_system",
        "n_completed_folds",
        "mean_val_mae",
        "mean_val_mse",
        "mean_val_weighted_mae",
        "mean_best_epoch",
        "mean_final_lr",
        "mean_runtime_sec",
    ]
    rows_dict = [dict(zip(cols, row)) for row in rows]
    if rows_dict:
        atomic_write_csv(path, cols, rows_dict)


# ============================================================
# 11. Manifest
# ============================================================
def build_manifest(folds: List[dict]) -> dict:
    combos = list(
        itertools.product(
            ARCHITECTURES,
            ACTIVATIONS,
            OPTIMIZERS,
            LR_POLICIES,
            LOSS_NAMES,
            FEATURE_SETS.keys(),
            NORMALIZATION_OPTIONS,
            UNIT_SYSTEMS,
        )
    )
    return {
        "data_root": str(DATA_ROOT.resolve()),
        "output_curves": OUTPUT_CURVES,
        "allowed_qs": ALLOWED_QS,
        "interior_qs": INTERIOR_QS,
        "edge_qs": EDGE_QS,
        "folds": folds,
        "architectures": ARCHITECTURES,
        "activations": ACTIVATIONS,
        "optimizers": OPTIMIZERS,
        "lr_policies": LR_POLICIES,
        "base_lr": BASE_LR,
        "loss_names": LOSS_NAMES,
        "feature_sets": FEATURE_SETS,
        "normalization_options": NORMALIZATION_OPTIONS,
        "unit_systems": UNIT_SYSTEMS,
        "max_epochs": MAX_EPOCHS,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "plateau_patience": PLATEAU_PATIENCE,
        "plateau_factor": PLATEAU_FACTOR,
        "min_lr": MIN_LR,
        "weighted_mae_alpha": WEIGHTED_MAE_ALPHA,
        "weighted_mae_power": WEIGHTED_MAE_POWER,
        "batch_size": BATCH_SIZE,
        "n_hparam_combos": len(combos),
        "n_total_runs": len(combos) * len(folds),
    }


# ============================================================
# 12. Main sweep
# ============================================================
def main() -> None:
    folds = build_6_folds()
    manifest = build_manifest(folds)
    atomic_write_text(MANIFEST_PATH, json.dumps(manifest, indent=2, ensure_ascii=False))

    q_data = build_q_curve_data(DATA_ROOT)

    log("Laddade data för q-värden: " + ", ".join(str(q) for q in sorted(q_data.keys())))
    for q in sorted(q_data.keys()):
        pack = q_data[q]
        log(
            f"q={q} | n_points_aug={len(pack.omega_mev)} | omega_min={pack.omega_mev.min():.6f} MeV | "
            f"omega_max={pack.omega_mev.max():.6f} MeV | step~{pack.inferred_step_mev:.6f} MeV"
        )

    conn = get_db_connection(DB_PATH)
    init_db(conn)

    log("Förbereder cache för fold-dataset ...")
    dm_cache = build_cached_fold_managers(q_data=q_data, folds=folds, device=DEVICE)
    log(f"Dataset-cache klar: {len(dm_cache)} kombinationer.")

    all_run_configs: List[RunConfig] = []
    for arch, activation, optimizer_name, lr_policy, loss_name, feature_set, normalize, unit_system in itertools.product(
        ARCHITECTURES,
        ACTIVATIONS,
        OPTIMIZERS,
        LR_POLICIES,
        LOSS_NAMES,
        FEATURE_SETS.keys(),
        NORMALIZATION_OPTIONS,
        UNIT_SYSTEMS,
    ):
        for fold in folds:
            all_run_configs.append(
                RunConfig(
                    fold_name=fold["fold_name"],
                    fold_index=fold["fold_index"],
                    val_q=fold["val_q"],
                    train_qs=fold["train_qs"],
                    architecture=list(arch),
                    activation=activation,
                    optimizer=optimizer_name,
                    lr_policy=lr_policy,
                    base_lr=BASE_LR,
                    loss_name=loss_name,
                    feature_set=feature_set,
                    normalize=normalize,
                    unit_system=unit_system,
                )
            )

    total_runs = len(all_run_configs)
    if MAX_RUNS is not None:
        all_run_configs = all_run_configs[:MAX_RUNS]
        total_runs = len(all_run_configs)

    log(f"Totalt antal körningar i svepet: {total_runs}")

    completed_before = conn.execute("SELECT COUNT(*) FROM runs WHERE status='completed'").fetchone()[0]
    log(f"Redan completed innan start: {completed_before}")

    started = time.time()
    executed_now = 0

    for idx, cfg in enumerate(all_run_configs, start=1):
        run_id = cfg.run_id()
        if run_completed(conn, run_id):
            continue

        run_seed = BASE_SEED + cfg.fold_index * 100000 + idx
        set_global_seed(run_seed)

        dm = dm_cache[(cfg.feature_set, bool(cfg.normalize), cfg.unit_system, cfg.fold_index)]

        upsert_run_start(
            conn,
            cfg,
            seed=run_seed,
            train_size=int(dm.X_train.shape[0]),
            val_size=int(dm.X_val.shape[0]),
        )

        try:
            result = train_one_run(dm, cfg)
            upsert_run_success(conn, cfg, result)

            if SAVE_TRAINING_HISTORY:
                history_path = HISTORY_DIR / f"history_{run_id}.json"
                atomic_write_text(history_path, json.dumps(result["history"], ensure_ascii=False))

            executed_now += 1
            elapsed = time.time() - started
            avg_per_new = elapsed / max(executed_now, 1)
            remaining_new = avg_per_new * max(total_runs - idx, 0)
            hh = int(remaining_new // 3600)
            mm = int((remaining_new % 3600) // 60)
            ss = int(remaining_new % 60)

            log(
                f"[{idx}/{total_runs}] completed | fold={cfg.fold_name} | val_q={cfg.val_q} | "
                f"arch={architecture_name(cfg.architecture)} | act={cfg.activation} | "
                f"sched={cfg.lr_policy} | loss={cfg.loss_name} | feats={cfg.feature_set} | "
                f"norm={cfg.normalize} | unit={cfg.unit_system} | "
                f"best_epoch={result['best_epoch']} | val_MAE={result['final_val_metrics']['mae']:.6e} | "
                f"val_MSE={result['final_val_metrics']['mse']:.6e} | "
                f"val_wMAE={result['final_val_metrics']['weighted_mae']:.6e} | ETA~{hh:02d}:{mm:02d}:{ss:02d}"
            )

        except Exception as exc:
            tb = traceback.format_exc()
            upsert_run_failure(conn, cfg, str(exc), tb)
            log(f"FAILED run_id={run_id} | fold={cfg.fold_name} | val_q={cfg.val_q} | error={exc}")
            log(tb)

        if executed_now > 0 and executed_now % EXPORT_EVERY_N_RUNS == 0:
            export_runs_csv(conn, RUNS_CSV_PATH)
            export_curve_csv(conn, CURVE_CSV_PATH)
            export_summary_csv(conn, SUMMARY_CSV_PATH)
            log("Exporterade delresultat till CSV.")

    export_runs_csv(conn, RUNS_CSV_PATH)
    export_curve_csv(conn, CURVE_CSV_PATH)
    export_summary_csv(conn, SUMMARY_CSV_PATH)

    n_completed = conn.execute("SELECT COUNT(*) FROM runs WHERE status='completed'").fetchone()[0]
    n_failed = conn.execute("SELECT COUNT(*) FROM runs WHERE status='failed'").fetchone()[0]
    total_elapsed = time.time() - started

    log(
        f"Klart. completed={n_completed}, failed={n_failed}, total_elapsed_sec={total_elapsed:.2f}. "
        f"Filer: {DB_PATH}, {RUNS_CSV_PATH}, {CURVE_CSV_PATH}, {SUMMARY_CSV_PATH}"
    )

    conn.close()


if __name__ == "__main__":
    main()
